### 첫번째 

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# 1. 데이터 로딩 & Merge
train       = pd.read_csv('../train.csv')
test        = pd.read_csv('../test.csv')
layout_info = pd.read_csv('../있는 layout_info.csv')

train = pd.merge(train, layout_info, on='layout_id', how='left')
test  = pd.merge(test,  layout_info, on='layout_id', how='left')
print(f"train: {train.shape}, test: {test.shape}")


# 2. 피처 엔지니어링
def make_features(df):
    df = df.copy()
    df = df.sort_values(['scenario_id', 'ID'])

    lag_cols = ['order_inflow_15m', 'congestion_score', 'robot_utilization',
                'robot_idle', 'low_battery_ratio', 'sku_concentration', 'avg_trip_distance']
    for col in lag_cols:
        df[f'{col}_lag1'] = df.groupby('scenario_id')[col].shift(1).fillna(df[col])
    df['order_trend']      = df['order_inflow_15m']  - df['order_inflow_15m_lag1']
    df['congestion_trend'] = df['congestion_score']  - df['congestion_score_lag1']
    df['trip_trend']       = df['avg_trip_distance'] - df['avg_trip_distance_lag1']

    robot_total = df['robot_active'] + df['robot_idle'] + df['robot_charging']
    df['robot_util_ratio']   = df['robot_active']   / (robot_total + 1)
    df['robot_idle_ratio']   = df['robot_idle']     / (robot_total + 1)
    df['robot_charge_ratio'] = df['robot_charging'] / (robot_total + 1)
    df['battery_health']     = df['battery_mean'] * (1 - df['low_battery_ratio'].fillna(0))
    df['charge_pressure']    = df['charge_queue_length'] / (df['charger_count'] + 1)
    df['fault_per_robot']    = df['fault_count_15m'].fillna(0) / (robot_total + 1)

    df['sku_x_trip']        = df['sku_concentration'] * df['avg_trip_distance']
    df['sku_x_congestion']  = df['sku_concentration'] * df['congestion_score'].fillna(0)
    df['trip_x_layout']     = df['avg_trip_distance'] * df['layout_compactness']
    df['trip_x_dispersion'] = df['avg_trip_distance'] * df['zone_dispersion']
    df['area_per_robot']    = df['floor_area_sqm']    / (df['robot_total'] + 1)
    df['charger_per_robot'] = df['charger_count']     / (df['robot_total'] + 1)
    df['aisle_x_oneway']    = df['aisle_width_avg']   * df['one_way_ratio']
    df['pack_x_inflow']     = df['pack_station_count'] * df['order_inflow_15m'].fillna(0)
    df['compactness_x_sku'] = df['layout_compactness'] * df['sku_concentration']

    df['order_complexity']    = df['order_inflow_15m'].fillna(0) * df['avg_items_per_order'].fillna(df['avg_items_per_order'].median())
    df['urgent_heavy']        = df['urgent_order_ratio'] * df['heavy_item_ratio'].fillna(0)
    df['congestion_pressure'] = df['order_inflow_15m'].fillna(0) * df['congestion_score'].fillna(0)

    df['shift_hour_sin']  = np.sin(2 * np.pi * df['shift_hour'] / 24)
    df['shift_hour_cos']  = np.cos(2 * np.pi * df['shift_hour'] / 24)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    df['env_stress'] = (
        df['ambient_noise_db'].fillna(0) / 100
        + df['floor_vibration_idx'].fillna(0)
        + (df['warehouse_temp_avg'].fillna(20) - 20).abs() / 20
    )
    return df

print("피처 엔지니어링 중...")
train = make_features(train)
test  = make_features(test)
print("완료!")


# 3. 결측치 처리
TARGET    = 'avg_delay_minutes_next_30m'
DROP_COLS = ['ID', 'layout_id', 'scenario_id', TARGET]

zero_cols = [col for col in train.columns if any(
    kw in col for kw in ['count', 'wait', 'queue', 'blocked', 'collision', 'fault']
)]
for col in zero_cols:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

numeric_cols = train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col not in zero_cols and col != TARGET:
        m_val = train[col].median()
        train[col] = train[col].fillna(m_val)
        test[col]  = test[col].fillna(m_val)

print(f"Train 결측치: {train.isnull().sum().sum()}")
print(f"Test 결측치: {test.isnull().sum().sum()}")


# 4. 인코딩 & 피처 준비
X      = train.drop(columns=DROP_COLS)
y      = train[TARGET]
X_test = test.drop(columns=[c for c in DROP_COLS if c != TARGET])

X      = pd.get_dummies(X, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

print(f"최종 피처 수: {X.shape[1]}")


# 5. LightGBM K-Fold 학습
params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,
    'num_leaves': 127,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_jobs': -1,
    'random_state': 42,
    'device': 'cpu',
}

kf   = KFold(n_splits=5, shuffle=True, random_state=42)
oof  = np.zeros(len(train))
pred = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**params, n_estimators=10000)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(300), lgb.log_evaluation(1000)]
    )

    oof[val_idx] = model.predict(X_val)
    pred        += model.predict(X_test) / 5

    mae = mean_absolute_error(y_val, oof[val_idx])
    print(f"Fold {fold+1} MAE: {mae:.4f}")

final_mae = mean_absolute_error(y, oof)
print(f"\n최종 OOF MAE: {final_mae:.4f}")


# 실험 결과 정리

## v2 실험 결과 (n_estimators=10000)

5-Fold Cross Validation을 적용하여 학습한 결과입니다.

Fold 1 MAE: 6.4916
Fold 2 MAE: 6.4147
Fold 3 MAE: 6.4208
Fold 4 MAE: 6.4635
Fold 5 MAE: 6.3725

최종 OOF MAE: 6.4164

5개의 Fold 중 가장 낮은 성능은 Fold 1의 6.4916이었고, 가장 높은 성능은 Fold 5의 6.3725였습니다. 5번의 평균을 낸 최종 OOF MAE는 6.4164로, 실제 딜레이와 예측값의 평균 오차가 약 6.41분이라는 의미입니다.


## v1과 v2 비교

v1은 n_estimators를 5000으로 설정했고 최종 OOF MAE는 6.7590이었습니다.

v2는 n_estimators를 10000으로 늘렸고 최종 OOF MAE는 6.4164로 개선되었습니다.

v1에서 학습이 끝났을 때 early stopping에 걸리지 않았습니다. 이는 5000번을 모두 돌렸는데도 모델이 아직 수렴하지 않았다는 뜻입니다. 그래서 n_estimators를 10000으로 두 배 늘렸고 MAE가 6.7590에서 6.4164로 0.34 개선되었습니다.

v2에서도 마찬가지로 early stopping에 걸리지 않았습니다. 따라서 n_estimators를 더 늘리면 추가 개선 가능성이 있습니다.


## 다음 실험 계획

n_estimators를 20000으로 늘려 추가 개선을 시도할 예정입니다. 이후 XGBoost와 앙상블을 구성하여 두 모델의 예측값을 평균내는 방식으로 성능을 높일 계획입니다.

### 두번째

In [2]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
import warnings
warnings.filterwarnings("ignore")

# 1. 데이터 로딩 & Merge
train       = pd.read_csv('../train.csv')
test        = pd.read_csv('../test.csv')
layout_info = pd.read_csv('../layout_info.csv')

train = pd.merge(train, layout_info, on='layout_id', how='left')
test  = pd.merge(test,  layout_info, on='layout_id', how='left')
print(f"train: {train.shape}, test: {test.shape}")


# 2. 피처 엔지니어링
def make_features(df):
    df = df.copy()
    df = df.sort_values(['scenario_id', 'ID'])

    # 시나리오 내 타임슬롯 순서 (1~25번째)
    df['timeslot_order'] = df.groupby('scenario_id').cumcount() + 1
    df['timeslot_ratio'] = df['timeslot_order'] / 25  # 시나리오 진행률 (0~1)
    df['is_early']  = (df['timeslot_order'] <= 5).astype(int)   # 초반 (1~5)
    df['is_middle'] = ((df['timeslot_order'] > 5) & (df['timeslot_order'] <= 15)).astype(int)  # 중반
    df['is_late']   = (df['timeslot_order'] > 15).astype(int)   # 후반 (16~25)

    # 시나리오 전체 통계 피처
    scenario_stats = df.groupby('scenario_id').agg(
        scenario_order_mean   = ('order_inflow_15m', 'mean'),
        scenario_order_max    = ('order_inflow_15m', 'max'),
        scenario_order_std    = ('order_inflow_15m', 'std'),
        scenario_congestion_mean = ('congestion_score', 'mean'),
        scenario_congestion_max  = ('congestion_score', 'max'),
        scenario_robot_mean   = ('robot_utilization', 'mean'),
        scenario_robot_min    = ('robot_utilization', 'min'),
        scenario_battery_mean = ('battery_mean', 'mean'),
        scenario_battery_min  = ('battery_mean', 'min'),
        scenario_fault_sum    = ('fault_count_15m', 'sum'),
    ).reset_index()
    df = df.merge(scenario_stats, on='scenario_id', how='left')

    # Lag 피처
    lag_cols = ['order_inflow_15m', 'congestion_score', 'robot_utilization',
                'robot_idle', 'low_battery_ratio', 'sku_concentration', 'avg_trip_distance']
    for col in lag_cols:
        df[f'{col}_lag1'] = df.groupby('scenario_id')[col].shift(1).fillna(df[col])
        df[f'{col}_lag2'] = df.groupby('scenario_id')[col].shift(2).fillna(df[col])
    df['order_trend']      = df['order_inflow_15m']  - df['order_inflow_15m_lag1']
    df['congestion_trend'] = df['congestion_score']  - df['congestion_score_lag1']
    df['trip_trend']       = df['avg_trip_distance'] - df['avg_trip_distance_lag1']

    # 현재값이 시나리오 평균 대비 얼마나 높은지
    df['order_vs_scenario_mean']     = df['order_inflow_15m'] - df['scenario_order_mean']
    df['congestion_vs_scenario_mean']= df['congestion_score'] - df['scenario_congestion_mean']

    # 로봇 관련
    robot_total = df['robot_active'] + df['robot_idle'] + df['robot_charging']
    df['robot_util_ratio']   = df['robot_active']   / (robot_total + 1)
    df['robot_idle_ratio']   = df['robot_idle']     / (robot_total + 1)
    df['robot_charge_ratio'] = df['robot_charging'] / (robot_total + 1)
    df['battery_health']     = df['battery_mean'] * (1 - df['low_battery_ratio'].fillna(0))
    df['charge_pressure']    = df['charge_queue_length'] / (df['charger_count'] + 1)
    df['fault_per_robot']    = df['fault_count_15m'].fillna(0) / (robot_total + 1)

    # 피처 중요도 기반 조합
    df['sku_x_trip']        = df['sku_concentration'] * df['avg_trip_distance']
    df['sku_x_congestion']  = df['sku_concentration'] * df['congestion_score'].fillna(0)
    df['trip_x_layout']     = df['avg_trip_distance'] * df['layout_compactness']
    df['trip_x_dispersion'] = df['avg_trip_distance'] * df['zone_dispersion']
    df['area_per_robot']    = df['floor_area_sqm']    / (df['robot_total'] + 1)
    df['charger_per_robot'] = df['charger_count']     / (df['robot_total'] + 1)
    df['aisle_x_oneway']    = df['aisle_width_avg']   * df['one_way_ratio']
    df['pack_x_inflow']     = df['pack_station_count'] * df['order_inflow_15m'].fillna(0)
    df['compactness_x_sku'] = df['layout_compactness'] * df['sku_concentration']

    # 주문 부하
    df['order_complexity']    = df['order_inflow_15m'].fillna(0) * df['avg_items_per_order'].fillna(df['avg_items_per_order'].median())
    df['urgent_heavy']        = df['urgent_order_ratio'] * df['heavy_item_ratio'].fillna(0)
    df['congestion_pressure'] = df['order_inflow_15m'].fillna(0) * df['congestion_score'].fillna(0)

    # 시간 주기성
    df['shift_hour_sin']  = np.sin(2 * np.pi * df['shift_hour'] / 24)
    df['shift_hour_cos']  = np.cos(2 * np.pi * df['shift_hour'] / 24)
    df['day_of_week_sin'] = np.sin(2 * np.pi * df['day_of_week'] / 7)
    df['day_of_week_cos'] = np.cos(2 * np.pi * df['day_of_week'] / 7)

    # 환경 스트레스
    df['env_stress'] = (
        df['ambient_noise_db'].fillna(0) / 100
        + df['floor_vibration_idx'].fillna(0)
        + (df['warehouse_temp_avg'].fillna(20) - 20).abs() / 20
    )
    return df

print("피처 엔지니어링 중...")
train = make_features(train)
test  = make_features(test)
print("완료!")


# 3. 결측치 처리
TARGET    = 'avg_delay_minutes_next_30m'
DROP_COLS = ['ID', 'layout_id', 'scenario_id', TARGET]

zero_cols = [col for col in train.columns if any(
    kw in col for kw in ['count', 'wait', 'queue', 'blocked', 'collision', 'fault']
)]
for col in zero_cols:
    train[col] = train[col].fillna(0)
    test[col]  = test[col].fillna(0)

numeric_cols = train.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    if col not in zero_cols and col != TARGET:
        m_val = train[col].median()
        train[col] = train[col].fillna(m_val)
        test[col]  = test[col].fillna(m_val)

print(f"Train 결측치: {train.isnull().sum().sum()}")
print(f"Test 결측치: {test.isnull().sum().sum()}")


# 4. 인코딩 & 피처 준비
X      = train.drop(columns=DROP_COLS)
y      = train[TARGET]
X_test = test.drop(columns=[c for c in DROP_COLS if c != TARGET])

X      = pd.get_dummies(X, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)
X, X_test = X.align(X_test, join='left', axis=1, fill_value=0)

print(f"최종 피처 수: {X.shape[1]}")


# 5. LightGBM K-Fold 학습
params = {
    'objective': 'regression_l1',
    'metric': 'mae',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': 0.01,
    'num_leaves': 127,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'n_jobs': -1,
    'random_state': 42,
    'device': 'cpu',
}

kf   = KFold(n_splits=5, shuffle=True, random_state=42)
oof  = np.zeros(len(train))
pred = np.zeros(len(test))

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    model = lgb.LGBMRegressor(**params, n_estimators=10000)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(300), lgb.log_evaluation(1000)]
    )

    oof[val_idx] = model.predict(X_val)
    pred        += model.predict(X_test) / 5

    mae = mean_absolute_error(y_val, oof[val_idx])
    print(f"Fold {fold+1} MAE: {mae:.4f}")

final_mae = mean_absolute_error(y, oof)
print(f"\n최종 OOF MAE: {final_mae:.4f}")


# # 6. 제출 파일 저장
# sub = pd.read_csv('sample_submission.csv')
# sub['avg_delay_minutes_next_30m'] = np.clip(pred, 0, None)
# sub.to_csv('submission.csv', index=False)

# print("submission.csv 저장 완료!")
# print(f"예측값 범위: {pred.min():.2f} ~ {pred.max():.2f}")
# print(f"예측값 평균: {pred.mean():.2f}")
# print(sub.head(10))

train: (250000, 108), test: (50000, 107)
피처 엔지니어링 중...
완료!
Train 결측치: 0
Test 결측치: 0
최종 피처 수: 163
Training until validation scores don't improve for 300 rounds
[1000]	valid_0's l1: 7.10632
[2000]	valid_0's l1: 6.46335
[3000]	valid_0's l1: 6.08261
[4000]	valid_0's l1: 5.84264
[5000]	valid_0's l1: 5.68321
[6000]	valid_0's l1: 5.59817
[7000]	valid_0's l1: 5.5316
[8000]	valid_0's l1: 5.49741
[9000]	valid_0's l1: 5.47361
[10000]	valid_0's l1: 5.43333
Did not meet early stopping. Best iteration is:
[10000]	valid_0's l1: 5.43333
Fold 1 MAE: 5.4333
Training until validation scores don't improve for 300 rounds
[1000]	valid_0's l1: 7.09352
[2000]	valid_0's l1: 6.44867
[3000]	valid_0's l1: 6.1241
[4000]	valid_0's l1: 5.86176
[5000]	valid_0's l1: 5.74548
[6000]	valid_0's l1: 5.67328
[7000]	valid_0's l1: 5.61971
[8000]	valid_0's l1: 5.54633
[9000]	valid_0's l1: 5.49559
[10000]	valid_0's l1: 5.45532
Did not meet early stopping. Best iteration is:
[10000]	valid_0's l1: 5.45532
Fold 2 MAE: 5.4553
Train

In [8]:
sub = pd.read_csv('../sample_submission.csv')
sub['avg_delay_minutes_next_30m'] = np.clip(pred, 0, None)
sub.to_csv('../submission.csv', index=False)

print("submission.csv 저장 완료!")
print(sub.head(10))

submission.csv 저장 완료!
            ID  avg_delay_minutes_next_30m
0  TEST_000000                   20.028624
1  TEST_000001                   28.260861
2  TEST_000002                   35.702904
3  TEST_000003                   39.223289
4  TEST_000004                   39.516015
5  TEST_000005                   37.842135
6  TEST_000006                   37.921740
7  TEST_000007                   37.742961
8  TEST_000008                   37.657757
9  TEST_000009                   36.081715


# 실험 결과 정리 v3

## 변경 사항

기존 코드에서 시나리오 구조를 활용한 피처를 추가했습니다.

데이터 설명을 보면 12,000개의 독립적인 시나리오가 있고 각 시나리오는 25개의 타임슬롯(15분 간격, 총 6시간)으로 구성되어 있습니다. 이 구조를 피처로 만들면 모델이 현재 시점이 시나리오의 어느 단계인지 파악할 수 있습니다.


## 추가된 피처

타임슬롯 순서 피처로 시나리오 내에서 현재가 몇 번째 타임슬롯인지(1~25)를 나타내는 timeslot_order와 시나리오 진행률을 0~1로 나타내는 timeslot_ratio를 추가했습니다. 또한 시나리오 초반(1~5번째), 중반(6~15번째), 후반(16~25번째)을 구분하는 is_early, is_middle, is_late 피처도 추가했습니다.

시나리오 전체 통계 피처로 같은 시나리오 내의 주문량 평균, 최대값, 표준편차, 혼잡도 평균과 최대값, 로봇 가동률 평균과 최솟값, 배터리 평균과 최솟값, 고장 횟수 합계를 추가했습니다.

현재값 대비 피처로 현재 주문량이 해당 시나리오 평균 대비 얼마나 높은지, 현재 혼잡도가 시나리오 평균 대비 얼마나 높은지를 나타내는 피처를 추가했습니다.

lag2 피처로 기존 lag1(15분 전)에 더해 lag2(30분 전) 값도 추가했습니다.


## 결과

최종 피처 수는 139개에서 163개로 늘었습니다.

Fold 1 MAE: 5.4333
Fold 2 MAE: 5.4147
Fold 3 MAE: 5.4521
Fold 4 MAE: 5.4371
Fold 5 MAE: 5.4663

최종 OOF MAE: 5.4439

v2의 6.4164에서 v3의 5.4439로 약 1점 가까이 개선되었습니다. 시나리오 구조를 활용한 피처가 큰 효과를 발휘했습니다.

이번에도 early stopping에 걸리지 않았으므로 n_estimators를 늘리면 추가 개선 가능성이 있습니다.